## 1. Load the team's V2 dataset

V2 already contains quality and language decisions from previous cleaning. **This notebook does not independently re-verify language, lyric–song correctness, or text copyright.** It only retains rows marked `KEEP`, which should be reviewed if the team's upstream criteria change.


In [ ]:
import csv, re, html, unicodedata, collections
from pathlib import Path
from google.colab import files

print("Please upload KM6312_Lyrics_Cleaned_v2.csv")
uploaded = files.upload()
file_candidates = [Path(n) for n in uploaded if n.lower().endswith('.csv')]
assert len(file_candidates) == 1, "Upload only the V2 CSV in this step."
input_path = file_candidates[0]

with input_path.open(encoding='utf-8-sig', newline='') as f:
    reader = csv.DictReader(f)
    source_rows = list(reader)
    source_columns = reader.fieldnames

print("Input rows:", len(source_rows))
print("Input columns:", len(source_columns))
assert len(source_rows) == 6075, "Unexpected row count: check the input version."
assert len(source_columns) == 46, "Unexpected input schema: check the V2 file."


## 2. Clean lyric text for a contextual emotion model

**BERT/DistilRoBERTa-friendly cleaning:** remove structural metadata like `[Verse 1]`, `[Chorus]`, URLs, trailing Genius `Embed` markers, and normalise whitespace. Preserve original word order, punctuation, capitalisation, negation (`not`, `never`, `don't`), and repeated lyric lines. Do **not** remove stopwords, stem, lemmatise, or truncate long lyrics here.

**Limitation:** This applies the same V3 cleaner used previously, including removal of **all** square-bracket spans; occasionally a bracket can contain actual sung lyrics. Such cases are flagged in the separate audit and require human review.


In [ ]:
SECTION = r'(?:pre[\s-]?chorus|post[\s-]?chorus|verse|chorus|bridge|intro|outro|hook|refrain|interlude|instrumental|guitar\s+solo|piano\s+solo|breakdown|spoken|sample|drop|break)'
ANY_BRACKET = re.compile(r'\[[^\]\n]{1,130}\]')
BRACKET_MALFORMED = re.compile(r'\[\s*' + SECTION + r'\b[^\}\n]{0,100}\}', re.I)
PAREN_TAG = re.compile(r'\(\s*' + SECTION + r'\b(?:\s*(?:[\w -]{1,30}|[:\-][^)]{0,50}))?\s*\)', re.I)
TRAILING_EMBED = re.compile(r'\s+(?:\d+\s*)?Embed\s*$', re.I)
URL = re.compile(r'https?://\S+|www\.\S+', re.I)
BOILERPLATE = re.compile(r'\b(?:You might also like|See upcoming pop shows|Get tickets as low as|Lyrics powered by|Contributors?\s*:)\b', re.I)
STRUCTURE_START = re.compile(r'^\s*' + SECTION + r'\b|^\s*(?:repeat|solo|pre-hook|pre-bridge)\b', re.I)
LYRIC_CLUE = re.compile(r"\b(?:i|you|we|my|your|love|don't|can't|won't|oh|yeah|hey|me|us|heart)\b", re.I)
WORD_PATTERN = re.compile(r"[^\W_]+(?:['’][^\W_]+)*", re.UNICODE)

def clean_for_bert(text):
    text = html.unescape(str(text or ''))
    text = unicodedata.normalize('NFKC', text)
    text = text.replace('\u200b', '').replace('\ufeff', '')
    text = ANY_BRACKET.sub(' ', text)
    text = BRACKET_MALFORMED.sub(' ', text)
    text = PAREN_TAG.sub(' ', text)
    text = TRAILING_EMBED.sub('', text)
    text = URL.sub(' ', text)
    return re.sub(r'\s+', ' ', text).strip()

def possible_bracketed_lyrics(text):
    spans = re.findall(r'\[([^\]\n]{1,130})\]', str(text or ''))
    return any(len(s.split()) >= 3 and not STRUCTURE_START.search(s)
               and LYRIC_CLUE.search(s) for s in spans)


## 3. Create the 12-column master table

- `song_id` = V2 `Source Row` (stable original chart-record ID, **not** a freshly assigned number).
- `original_lyrics` = exact value from V2 `Lyrics`, kept for audit.
- `bert_lyrics` = the cleaned text from the function above.
- `word_count`, `unique_word_count`, `lexical_diversity` = newly calculated from **bert_lyrics**, based on Unicode-aware word tokens; uniqueness is case-insensitive. These may not equal the old EDA fields because the counting definition is different.
- `year`, `rank`, `decade` = year-end Billboard chart attributes. A song that charted in two years remains two records; do **not** use both records across train/test splits.
- `language` = `en` for records accepted as English in V2, including manually review-recovered entries.


In [ ]:
HEADERS = ['song_id', 'song_title', 'artist', 'year', 'rank',
           'original_lyrics', 'bert_lyrics', 'word_count',
           'unique_word_count', 'lexical_diversity', 'decade', 'language']

master = []
review = []
removed = 0
structure_tags_removed = 0

for row in source_rows:
    # Keep V2-approved records; this project version already has 6,075 KEEP rows.
    if row['Quality Decision v2'] != 'KEEP':
        removed += 1
        continue
    song_id = row['Source Row'].strip()
    cleaned = clean_for_bert(row['Lyrics'])
    if not cleaned:
        removed += 1
        continue

    language_decision = row['Language Decision v2'].strip()
    if not language_decision.startswith('en ('):
        raise ValueError(f'Unverified English language at song_id={song_id}')

    tokens = WORD_PATTERN.findall(cleaned)
    word_count = len(tokens)
    unique_word_count = len({word.casefold() for word in tokens})
    year = int(row['Year'])

    master.append({
        'song_id': song_id,
        'song_title': row['Song Title'].strip(),
        'artist': row['Artist'].strip(),
        'year': year,
        'rank': int(row['Rank']),
        'original_lyrics': row['Lyrics'],
        'bert_lyrics': cleaned,
        'word_count': word_count,
        'unique_word_count': unique_word_count,
        'lexical_diversity': f'{unique_word_count/word_count:.6f}' if word_count else '',
        'decade': (year // 10) * 10,
        'language': 'en'
    })

    structure_tags_removed += (len(ANY_BRACKET.findall(row['Lyrics'])) +
                               len(BRACKET_MALFORMED.findall(row['Lyrics'])) +
                               len(PAREN_TAG.findall(row['Lyrics'])))
    reasons = []
    if possible_bracketed_lyrics(row['Lyrics']):
        reasons.append('Bracketed text may have contained actual sung lyrics')
    if 'REVIEW NOTE' in row['Quality Reason v2']:
        reasons.append('V2 flagged this record for review')
    if word_count < 40:
        reasons.append('Fewer than 40 words')
    if BOILERPLATE.search(cleaned):
        reasons.append('Possible website boilerplate')
    if reasons:
        review.append({'song_id': song_id, 'song_title': row['Song Title'],
                       'artist': row['Artist'], 'year': year,
                       'reason': '; '.join(reasons)})

print(f'Input rows: {len(source_rows):,}')
print(f'Retained: {len(master):,}; removed this step: {removed}')
print(f'Columns: {len(HEADERS)}')
print(f'Potential manual-review rows: {len(review)}')
print(f'Bracket/structural items removed: {structure_tags_removed:,}')


## 4. Quality control and export

The 12-column file is the **team-level analysis master**. Keep the unmodified V2 as an archived source. Cleaning issues are kept in a separate review CSV instead of adding audit columns to the master. The 42 flagged songs **remain** in the master until manually reviewed.


In [ ]:
assert len(master) == 6075, 'Row count differs from expected 6,075.'
assert len({r['song_id'] for r in master}) == len(master), 'Duplicate song IDs detected.'
assert len(HEADERS) == 12
assert all(r['original_lyrics'] and r['bert_lyrics'] for r in master)
assert all(r['unique_word_count'] <= r['word_count'] for r in master)
assert all(r['language'] == 'en' for r in master)
assert all(r['decade'] == (r['year']//10)*10 for r in master)

master_path = Path('KM6312_Master_Dataset_12col.csv')
review_path = Path('KM6312_Master_Quality_Review.csv')

with master_path.open('w', encoding='utf-8-sig', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=HEADERS)
    writer.writeheader()
    writer.writerows(master)

with review_path.open('w', encoding='utf-8-sig', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['song_id','song_title','artist','year','reason'])
    writer.writeheader()
    writer.writerows(review)

# Preview metadata only (avoid printing full copyrighted lyrics).
for r in master[:5]:
    print(r['song_id'], r['song_title'], r['year'],
          r['word_count'], r['unique_word_count'], r['decade'])
print('Saved:', master_path, master_path.stat().st_size, 'bytes')
print('Saved:', review_path, review_path.stat().st_size, 'bytes')

files.download(str(master_path))
files.download(str(review_path))


## 5. Dataset usage notes for the group

**EDA:** `year`, `rank`, `word_count`, `unique_word_count`, `lexical_diversity`, `decade`.

**LDA:** use `bert_lyrics` as a source but create an **LDA-specific tokenised/stopword-filtered corpus** separately. Keep the same `song_id` for merging.

**Emotion Classification:** use `bert_lyrics` without removing stopwords or negation. Note that transformer models may process only the first 512 tokens unless a justified chunking strategy is applied. **This notebook does not truncate lyrics.**

**Regression:** use `song_id`, `year`, `rank`, and merge emotion predictions and LDA topic probabilities by `song_id`.

**Important:** These 6,075 records are annual chart rows, not necessarily distinct songs. Split supervised training/test data by song or lyric hash to prevent leakage.

### Data preprocessing write-up (for report)

> We started from the previously cleaned English-lyrics dataset (V2; 6,075 chart records and 46 columns). For downstream analysis, we created a compact 12-column master table. Song identity, chart year, ranking, and the original lyrics were retained. A BERT-ready lyric field was constructed by removing structural annotations and non-lyrical URL/embedding artifacts while preserving word order, negation, punctuation, and repetitions. Word count, case-insensitive unique word count, and lexical diversity were recomputed from this cleaned lyric field, and decade was derived from chart year. No additional chart records were discarded. A separate quality-review file was generated for potentially ambiguous lyric cleaning or previous review notes. The master retains stable source-row identifiers, enabling alignment with LDA, emotion classification, and regression results.
